In [5]:
from matplotlib import cm
import numpy as np

In [1]:
import matplotlib
matplotlib.use("QtAgg")   # non-interactive, no windows


In [2]:
import scipy.io as sio
from pathlib import Path
import matplotlib.pyplot as plt

folder = Path(r"F:\INTRSECT_INVIVO\Analysis")

mat_files = list(folder.glob("*.mat"))

# If you want strings instead of Path objects:
mat_files = [str(f) for f in mat_files]

print(mat_files)


['F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV1_T1.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV1_T2.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV1_T3.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV1_T4.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV1_T5.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV1_T6.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV2_T1.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV2_T2.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV2_T3.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV2_T4.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251127-FOV2_T5.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251211-FOV1_T1.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251211-FOV1_T2.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251211-FOV1_T3.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1L-20251211-FOV1_T4.mat', 'F:\\INTRSECT_INVIVO\\Analysis\\NF170.1

In [3]:
len(mat_files)

314

In [6]:
matname=mat_files[187
]
volpy_data = sio.loadmat(
    matname,
    squeeze_me=True,
    struct_as_record=False
)

vpy = volpy_data['vpy']

ROIs = vpy.ROIs
img  = vpy.img

#display 1st an d3rd channel of img side by side
plt.figure(figsize=(10,5))
plt.subplot(1,3,1)
plt.imshow(img[:,:,0], cmap='gray')
plt.title("Channel 1")

plt.subplot(1,3,2)
plt.imshow(img[:,:,2], cmap='gray')
plt.title("Channel 3")

#overlay ROIs on full img on right
plt.subplot(1,3,3)
plt.imshow(img, cmap='gray')

# pick a colormap with many distinct colors
cmap = cm.get_cmap('tab20', len(ROIs))

for i, roi in enumerate(ROIs):
    mask = np.squeeze(roi).astype(bool)

    color = cmap(i)          # RGBA tuple
    rgba = np.zeros((*mask.shape, 4))
    rgba[mask] = [*color[:3], 0.35]   # keep alpha fixed

    plt.imshow(rgba)

plt.title("ROIs Overlay")
plt.show()

C:\Users\ICNLab\AppData\Local\Temp\ipykernel_13592\1342554734.py:29: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('tab20', len(ROIs))


In [11]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from matplotlib.colors import hsv_to_rgb
from PIL import Image


# for files in TrainingDataFolder = r'C:\Users\ICNLab\caiman_data\Training_Data'  ending in .npy set MASK_FILE= that path, then find the single png file whose date time matches the datetime in the npy file's name.  The npy file is called r'C:\Users\ICNLab\caiman_data\Training_Data\'

# MASK_FILE = "XXXX.npy"
# IMG_FILE = "YYY.png"

# ----------------------------
# Load data
# ----------------------------
masks = ROIs #np.load(MASK_FILE)  # (H,W,N)
H, W, N = masks.shape

#img = np.array(Image.open(IMG_FILE).convert("L"))
p1, p99 = np.percentile(img, (1, 99))
img = np.clip((img - p1) / (p99 - p1), 0, 1)

# ----------------------------
# Color handling
# ----------------------------
def generate_colors(n):
    return hsv_to_rgb(
        np.column_stack([
            np.linspace(0, 1, n, endpoint=False),
            np.ones(n),
            np.ones(n)
        ])
    )

colors = generate_colors(N)
ALPHA = 0.35

# ----------------------------
# Main viewer
# ----------------------------
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_title("Cell Mask Viewer")

# define extent so that x=0..W, y=0..H
extent = [0, W, H, 0]  # top-left origin like MATLAB

# main image
ax.imshow(img, cmap='gray', extent=extent, interpolation='none')

# overlay
overlay = np.zeros((H, W, 4), dtype=float)
im_overlay = ax.imshow(overlay, extent=extent, interpolation='none')

# fix axes to match image pixels exactly
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.set_aspect('equal')
ax.axis('off')



hovered = None
create_mode = {"active": False}

# ----------------------------
# Overlay redraw
# ----------------------------
# redraw function
def redraw_overlay(highlight=None):
    overlay[:] = 0
    for i in range(masks.shape[2]):
        mask = masks[:, :, i]
        overlay[mask > 0, :3] = colors[i]
        overlay[mask > 0, 3] = ALPHA
    if highlight is not None:
        overlay[masks[:, :, highlight] > 0, 3] = 0.8
    im_overlay.set_data(overlay)
    fig.canvas.draw_idle()


redraw_overlay()

# ----------------------------
# Hover logic
# ----------------------------
def on_move(event):
    global hovered
    if event.inaxes != ax or create_mode["active"]:
        return
    x, y = int(event.xdata), int(event.ydata)
    if x < 0 or y < 0 or x >= W or y >= H:
        return
    hits = np.where(masks[y, x, :] > 0)[0]
    if len(hits):
        if hovered != hits[0]:
            hovered = hits[0]
            redraw_overlay(hovered)
    else:
        if hovered is not None:
            hovered = None
            redraw_overlay()

# ----------------------------
# Click logic
# ----------------------------
def on_click(event):
    global masks, colors
    if event.inaxes != ax:
        return

    x, y = int(event.xdata), int(event.ydata)

    if create_mode["active"]:
        # Create new empty mask
        new_mask = np.zeros((H, W), dtype=bool)
        masks = np.dstack([masks, new_mask])
        colors = generate_colors(masks.shape[2])
        idx = masks.shape[2] - 1
        create_mode["active"] = False
        open_editor(idx, center=(y, x))
        redraw_overlay()
        return

    if hovered is not None:
        open_editor(hovered)

fig.canvas.mpl_connect("motion_notify_event", on_move)
fig.canvas.mpl_connect("button_press_event", on_click)

# ----------------------------
# New mask button
# ----------------------------
new_ax = plt.axes([0.01, 0.01, 0.18, 0.06])
new_btn = Button(new_ax, "New Mask")

def activate_new_mask(event):
    create_mode["active"] = True
    ax.set_title("Click anywhere to create a new mask")

new_btn.on_clicked(activate_new_mask)

# ----------------------------
# Editor window
# ----------------------------
def open_editor(idx, center=None):
    global masks

    mask = masks[:, :, idx]

    if center is None and mask.any():
        ys, xs = np.where(mask)
        cy, cx = int(np.mean(ys)), int(np.mean(xs))
    else:
        cy, cx = center if center else (H // 2, W // 2)

    r = 20
    y0, y1 = max(0, cy - r), min(H, cy + r)
    x0, x1 = max(0, cx - r), min(W, cx + r)

    sub_img = img[y0:y1, x0:x1]
    sub_mask = mask[y0:y1, x0:x1].copy()

    fig2, ax2 = plt.subplots()
    ax2.set_title(f"Editing mask {idx} (Draw)")
    ax2.imshow(sub_img, cmap="gray")
    mask_im = ax2.imshow(sub_mask, cmap="Reds", alpha=0.5)

    brush_ax = plt.axes([0.25, 0.02, 0.35, 0.03])
    brush_slider = Slider(brush_ax, "Brush", 1, 10, valinit=3)

    save_ax = plt.axes([0.63, 0.02, 0.15, 0.05])
    save_btn = Button(save_ax, "Save")

    del_ax = plt.axes([0.8, 0.02, 0.18, 0.05])
    del_btn = Button(del_ax, "Delete")

    mode = {"erase": False}

    def draw(event):
        if event.inaxes != ax2:
            return
        x, y = int(event.xdata), int(event.ydata)
        rr = int(brush_slider.val)
        yy, xx = np.ogrid[-rr:rr+1, -rr:rr+1]
        circle = xx**2 + yy**2 <= rr**2

        ys = slice(max(0, y-rr), min(sub_mask.shape[0], y+rr+1))
        xs = slice(max(0, x-rr), min(sub_mask.shape[1], x+rr+1))
        cm = circle[:ys.stop-ys.start, :xs.stop-xs.start]

        if mode["erase"]:
            sub_mask[ys, xs][cm] = 0
        else:
            sub_mask[ys, xs][cm] = 1

        mask_im.set_data(sub_mask)
        fig2.canvas.draw_idle()

    def toggle_mode(event):
        mode["erase"] = not mode["erase"]
        ax2.set_title(f"Editing mask {idx} ({'Erase' if mode['erase'] else 'Draw'})")

    def save(event):
        masks[y0:y1, x0:x1, idx] = sub_mask
        redraw_overlay()
        plt.close(fig2)

    def delete(event):
        nonlocal idx
        masks = np.delete(masks, idx, axis=2)
        redraw_overlay()
        plt.close(fig2)

    fig2.canvas.mpl_connect("motion_notify_event", draw)
    fig2.canvas.mpl_connect("button_press_event", toggle_mode)
    save_btn.on_clicked(save)
    del_btn.on_clicked(delete)

    plt.show()

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from matplotlib.colors import hsv_to_rgb


# ----------------------------
# Example inputs
# ----------------------------
# ROIs: (N, H, W)
# img: (H, W, 3)
# Replace these with your actual data
N = ROIs.shape[0]
H, W = 512, 512
masks = ROIs  # shape: (N, H, W)
img_rgb = img  # shape: (H, W, 3), values in 0..1

# ----------------------------
# Generate colors
# ----------------------------
def generate_colors(n):
    return hsv_to_rgb(
        np.column_stack([
            np.linspace(0, 1, n, endpoint=False),
            np.ones(n),
            np.ones(n)
        ])
    )

colors = generate_colors(N)
ALPHA = 0.35

# ----------------------------
# Main viewer
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_title("Cell Mask Viewer")

# Display the main image
ax.imshow(img_rgb, extent=[0, W, H, 0], interpolation='none')
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.set_aspect('equal')
ax.axis('off')

# Overlay: RGBA array
overlay = np.zeros((H, W, 4), dtype=float)
im_overlay = ax.imshow(overlay, extent=[0, W, H, 0], interpolation='none')

hovered = None
create_mode = {"active": False}

# ----------------------------
# Overlay redraw
# ----------------------------
def redraw_overlay(highlight=None):
    overlay[:] = 0
    for i in range(N):
        mask = masks[i]  # (H, W)
        overlay[mask > 0, :3] = colors[i]
        overlay[mask > 0, 3] = ALPHA
    if highlight is not None:
        overlay[masks[highlight] > 0, 3] = 0.8
    im_overlay.set_data(overlay)
    fig.canvas.draw_idle()

redraw_overlay()

# ----------------------------
# Hover logic
# ----------------------------
def on_move(event):
    global hovered
    if event.inaxes != ax or create_mode["active"]:
        return
    x, y = int(event.xdata), int(event.ydata)
    if x < 0 or y < 0 or x >= W or y >= H:
        return
    hits = np.where([mask[y, x] for mask in masks])[0]
    if len(hits):
        if hovered != hits[0]:
            hovered = hits[0]
            redraw_overlay(hovered)
    else:
        if hovered is not None:
            hovered = None
            redraw_overlay()

# ----------------------------
# Click logic
# ----------------------------
def on_click(event):
    global masks, colors, N
    if event.inaxes != ax:
        return

    x, y = int(event.xdata), int(event.ydata)

    if create_mode["active"]:
        # Create new empty mask
        new_mask = np.zeros((H, W), dtype=bool)
        masks = np.concatenate([masks, new_mask[None, :, :]], axis=0)
        N = masks.shape[0]
        colors = generate_colors(N)
        create_mode["active"] = False
        redraw_overlay()
        return

fig.canvas.mpl_connect("motion_notify_event", on_move)
fig.canvas.mpl_connect("button_press_event", on_click)

# ----------------------------
# New mask button
# ----------------------------
new_ax = plt.axes([0.01, 0.01, 0.18, 0.06])
new_btn = Button(new_ax, "New Mask")

def activate_new_mask(event):
    create_mode["active"] = True
    ax.set_title("Click anywhere to create a new mask")

new_btn.on_clicked(activate_new_mask)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from matplotlib.colors import hsv_to_rgb

# ----------------------------
# Example inputs
# ----------------------------
# ROIs: (N, H, W)
# img: (H, W, 3)
masks = ROIs.copy()  # (N, 512, 512)
img_rgb = img         # (512, 512, 3)
N, H, W = masks.shape

# ----------------------------
# Generate colors
# ----------------------------
def generate_colors(n):
    return hsv_to_rgb(np.column_stack([
        np.linspace(0, 1, n, endpoint=False),
        np.ones(n),
        np.ones(n)
    ]))

colors = generate_colors(N)
ALPHA = 0.35

# ----------------------------
# Main viewer
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_title("Cell Mask Viewer")

extent = [0, W, H, 0]  # top-left origin

ax.imshow(img_rgb, extent=extent, interpolation='none')
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.set_aspect('equal')
ax.axis('off')

# Overlay
overlay = np.zeros((H, W, 4), dtype=float)
im_overlay = ax.imshow(overlay, extent=extent, interpolation='none')

hovered = None
create_mode = {"active": False}

# ----------------------------
# Overlay redraw
# ----------------------------
def redraw_overlay(highlight=None):
    overlay[:] = 0
    for i in range(masks.shape[0]):
        mask = masks[i]
        overlay[mask > 0, :3] = colors[i]
        overlay[mask > 0, 3] = ALPHA
    if highlight is not None:
        overlay[masks[highlight] > 0, 3] = 0.8
    im_overlay.set_data(overlay)
    fig.canvas.draw_idle()

redraw_overlay()

# ----------------------------
# Hover logic
# ----------------------------
def on_move(event):
    global hovered
    if event.inaxes != ax or create_mode["active"]:
        return
    x, y = int(event.xdata), int(event.ydata)
    if x < 0 or y < 0 or x >= W or y >= H:
        return
    hits = np.where([mask[y, x] for mask in masks])[0]
    if len(hits):
        if hovered != hits[0]:
            hovered = hits[0]
            redraw_overlay(hovered)
    else:
        if hovered is not None:
            hovered = None
            redraw_overlay()

# ----------------------------
# Click logic
# ----------------------------
def on_click(event):
    global masks, colors
    if event.inaxes != ax:
        return

    x, y = int(event.xdata), int(event.ydata)

    if create_mode["active"]:
        # Create new empty mask
        new_mask = np.zeros((H, W), dtype=bool)
        masks = np.concatenate([masks, new_mask[None, :, :]], axis=0)
        colors = generate_colors(masks.shape[0])
        idx = masks.shape[0]-1
        create_mode["active"] = False
        open_editor(idx, center=(y, x))
        redraw_overlay()
        return

    if hovered is not None:
        open_editor(hovered)

fig.canvas.mpl_connect("motion_notify_event", on_move)
fig.canvas.mpl_connect("button_press_event", on_click)

# ----------------------------
# New mask button
# ----------------------------
new_ax = plt.axes([0.01, 0.01, 0.18, 0.06])
new_btn = Button(new_ax, "New Mask")
new_btn.on_clicked(lambda event: activate_new_mask(event := event))

def activate_new_mask(event):
    create_mode["active"] = True
    ax.set_title("Click anywhere to create a new mask")

# ----------------------------
# Editor
# ----------------------------
def open_editor(idx, center=None):
    global masks
    mask = masks[idx]

    # center
    if center is None and mask.any():
        ys, xs = np.where(mask)
        cy, cx = int(np.mean(ys)), int(np.mean(xs))
    else:
        cy, cx = center if center else (H//2, W//2)

    # crop region
    r = 30
    y0, y1 = max(0, cy-r), min(H, cy+r)
    x0, x1 = max(0, cx-r), min(W, cx+r)
    sub_img = img_rgb[y0:y1, x0:x1]
    sub_mask = mask[y0:y1, x0:x1].copy()

   # Sub-image editor
    fig2, ax2 = plt.subplots()
    fig2.subplots_adjust(bottom=0.15)  # leave space for buttons

    ax2.imshow(sub_img, interpolation='none')
    mask_im = ax2.imshow(sub_mask, cmap='Reds', alpha=0.5, interpolation='none')
    ax2.axis('off')

    # Buttons
    save_ax = plt.axes([0.7, 0.02, 0.12, 0.05])
    save_btn = Button(save_ax, "Save")

    del_ax = plt.axes([0.83, 0.02, 0.12, 0.05])
    del_btn = Button(del_ax, "Delete")

    toggle_ax = plt.axes([0.01, 0.02, 0.12, 0.05])
    toggle_btn = Button(toggle_ax, "Erase/Draw")


    mode = {"erase": False}

    def draw(event):
        if not drawing["active"]:
            return
        if event.inaxes != ax2:
            return

        x, y = int(event.xdata), int(event.ydata)
        rr = int(brush_slider.val)
        yy, xx = np.ogrid[-rr:rr+1, -rr:rr+1]
        circle = xx**2 + yy**2 <= rr**2

        ys_slice = slice(max(0, y-rr), min(sub_mask.shape[0], y+rr+1))
        xs_slice = slice(max(0, x-rr), min(sub_mask.shape[1], x+rr+1))
        cm = circle[:ys_slice.stop-ys_slice.start, :xs_slice.stop-xs_slice.start]

        if mode["erase"]:
            sub_mask[ys_slice, xs_slice][cm] = 0
        else:
            sub_mask[ys_slice, xs_slice][cm] = 1

        mask_im.set_data(sub_mask)
        fig2.canvas.draw_idle()


    def toggle_mode(event):
        # Press 'e' to toggle erase/draw mode
        if event.key != 'e':
            return
        mode["erase"] = not mode["erase"]
        ax2.set_title(f"Editing mask {idx} ({'Erase' if mode['erase'] else 'Draw'})")


    # Track if mouse button is pressed
    drawing = {"active": False}

    def on_press(event):
        if event.inaxes != ax2:
            return
        drawing["active"] = True
        draw(event)  # draw immediately

    def on_release(event):
        drawing["active"] = False

    # Connect events
    fig2.canvas.mpl_connect("button_press_event", on_press)
    fig2.canvas.mpl_connect("button_release_event", on_release)
    fig2.canvas.mpl_connect("motion_notify_event", draw)
    fig2.canvas.mpl_connect("key_press_event", toggle_mode)



    def save(event):
        masks[idx][y0:y1, x0:x1] = sub_mask
        redraw_overlay()
        plt.close(fig2)

    def delete(event):
        nonlocal idx
        masks = np.delete(masks, idx, axis=0)
        redraw_overlay()
        plt.close(fig2)


    save_btn.on_clicked(save)
    del_btn.on_clicked(delete)

    plt.show()

plt.show()


Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\matplotlib\cbook.py", line 298, in process
    func(*args, **kwargs)
  File "C:\Users\ICNLab\AppData\Local\Temp\ipykernel_29112\3757580138.py", line 204, in on_press
    draw(event)  # draw immediately
  File "C:\Users\ICNLab\AppData\Local\Temp\ipykernel_29112\3757580138.py", line 172, in draw
    rr = int(brush_slider.val)
NameError: name 'brush_slider' is not defined
Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\matplotlib\cbook.py", line 298, in process
    func(*args, **kwargs)
  File "C:\Users\ICNLab\AppData\Local\Temp\ipykernel_29112\3757580138.py", line 172, in draw
    rr = int(brush_slider.val)
NameError: name 'brush_slider' is not defined
Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\matplotlib\cbook.py", line 298, in process
    func(*args, **kwargs)
  File "C:\Users\ICNLab

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from matplotlib.colors import hsv_to_rgb

# ----------------------------
# Example Data Setup (Placeholders)
# ----------------------------
# Ensure ROIs and img are defined before this script runs
# For testing: 
# ROIs = np.zeros((5, 512, 512), dtype=bool)
# img = np.random.rand(512, 512, 3)

masks = ROIs.copy().astype(bool)
img_rgb = img
N, H, W = masks.shape

# ----------------------------
# Helper Functions
# ----------------------------
def generate_colors(n):
    if n == 0: return np.array([])
    return hsv_to_rgb(np.column_stack([
        np.linspace(0, 1, n, endpoint=False),
        np.ones(n),
        np.ones(n)
    ]))

colors = generate_colors(N)
ALPHA = 0.35

# ----------------------------
# Main Viewer Setup
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 8))
plt.subplots_adjust(bottom=0.1)
ax.set_title("Cell Mask Viewer")

extent = [0, W, H, 0]
ax.imshow(img_rgb, extent=extent, interpolation='none')
overlay = np.zeros((H, W, 4), dtype=float)
im_overlay = ax.imshow(overlay, extent=extent, interpolation='none', zorder=10)

ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.axis('off')

hovered = None
create_mode = {"active": False}

def redraw_overlay(highlight=None):
    global colors
    overlay[:] = 0
    # Update colors in case N changed
    if len(colors) != masks.shape[0]:
        colors = generate_colors(masks.shape[0])
        
    for i in range(masks.shape[0]):
        m = masks[i]
        overlay[m > 0, :3] = colors[i]
        overlay[m > 0, 3] = 0.8 if i == highlight else ALPHA
    
    im_overlay.set_data(overlay)
    fig.canvas.draw_idle()

# ----------------------------
# Interaction Logic
# ----------------------------
def on_move(event):
    global hovered
    if event.inaxes != ax or create_mode["active"]:
        return
    x, y = int(event.xdata), int(event.ydata)
    if not (0 <= x < W and 0 <= y < H):
        return
    
    # Check masks from top to bottom
    hits = np.where([masks[i, y, x] for i in range(len(masks))])[0]
    new_hover = hits[-1] if len(hits) > 0 else None
    
    if new_hover != hovered:
        hovered = new_hover
        redraw_overlay(hovered)

def on_click(event):
    global masks, colors
    if event.inaxes != ax: return

    x, y = int(event.xdata), int(event.ydata)

    if create_mode["active"]:
        new_mask = np.zeros((H, W), dtype=bool)
        masks = np.concatenate([masks, new_mask[None, :, :]], axis=0)
        colors = generate_colors(masks.shape[0])
        create_mode["active"] = False
        ax.set_title("Cell Mask Viewer")
        open_editor(len(masks) - 1, center=(y, x))
        return

    if hovered is not None:
        open_editor(hovered)

fig.canvas.mpl_connect("motion_notify_event", on_move)
fig.canvas.mpl_connect("button_press_event", on_click)

# ----------------------------
# New Mask Button
# ----------------------------
new_ax = plt.axes([0.4, 0.02, 0.2, 0.05])
new_btn = Button(new_ax, "Add New Mask")

def activate_new_mask(event):
    create_mode["active"] = True
    ax.set_title("CLICK ON IMAGE TO PLACE NEW MASK")
    fig.canvas.draw_idle()

new_btn.on_clicked(activate_new_mask)

# ----------------------------
# Editor Window
# ----------------------------
# Store references to prevent garbage collection
editor_widgets = [] 

def open_editor(idx, center=None):
    global masks, colors
    mask = masks[idx]

    if center is None and mask.any():
        ys, xs = np.where(mask)
        cy, cx = int(np.mean(ys)), int(np.mean(xs))
    else:
        cy, cx = center if center else (H//2, W//2)

    r = 40
    y0, y1 = max(0, cy-r), min(H, cy+r)
    x0, x1 = max(0, cx-r), min(W, cx+r)
    
    sub_img = img_rgb[y0:y1, x0:x1]
    sub_mask = mask[y0:y1, x0:x1].copy()

    fig2, ax2 = plt.subplots(figsize=(6, 6))
    fig2.subplots_adjust(bottom=0.2)
    ax2.imshow(sub_img)
    mask_im = ax2.imshow(sub_mask, cmap='Reds', alpha=0.4, interpolation='none')
    ax2.set_title(f"Editing Mask {idx} - 'e' to toggle Erase/Draw")

    # Persistence list for this window's widgets
    current_widgets = []

    state = {"drawing": False, "erase": False, "brush": 2}

    def update_mask(event):
        if not state["drawing"] or event.inaxes != ax2: return
        ix, iy = int(event.xdata), int(event.ydata)
        
        # Simple brush logic
        yy, xx = np.ogrid[-state["brush"]:state["brush"]+1, -state["brush"]:state["brush"]+1]
        circle = xx**2 + yy**2 <= state["brush"]**2
        
        sy0, sy1 = max(0, iy-state["brush"]), min(sub_mask.shape[0], iy+state["brush"]+1)
        sx0, sx1 = max(0, ix-state["brush"]), min(sub_mask.shape[1], ix+state["brush"]+1)
        
        # Slice the circle to fit boundaries
        c_slice = circle[0:(sy1-sy0), 0:(sx1-sx0)]
        sub_mask[sy0:sy1, sx0:sx1][c_slice] = not state["erase"]
        
        mask_im.set_data(sub_mask)
        fig2.canvas.draw_idle()

    def on_p(event): state["drawing"] = True; update_mask(event)
    def on_r(event): state["drawing"] = False
    def on_k(event):
        if event.key == 'e':
            state["erase"] = not state["erase"]
            ax2.set_title(f"Mode: {'ERASING' if state['erase'] else 'DRAWING'}")
            fig2.canvas.draw_idle()

    fig2.canvas.mpl_connect("button_press_event", on_p)
    fig2.canvas.mpl_connect("button_release_event", on_r)
    fig2.canvas.mpl_connect("motion_notify_event", update_mask)
    fig2.canvas.mpl_connect("key_press_event", on_k)

    # Save Button
    s_ax = plt.axes([0.1, 0.05, 0.25, 0.075])
    s_btn = Button(s_ax, "Save")
    def save_cb(event):
        masks[idx][y0:y1, x0:x1] = sub_mask
        redraw_overlay()
        plt.close(fig2)
    s_btn.on_clicked(save_cb)
    current_widgets.append(s_btn)

    # Delete Button
    d_ax = plt.axes([0.65, 0.05, 0.25, 0.075])
    d_btn = Button(d_ax, "Delete", color='salmon')
    def del_cb(event):
        global masks, colors
        masks = np.delete(masks, idx, axis=0)
        colors = generate_colors(len(masks))
        redraw_overlay()
        plt.close(fig2)
    d_btn.on_clicked(del_cb)
    current_widgets.append(d_btn)
    
    editor_widgets.append(current_widgets)
    plt.show()

redraw_overlay()
plt.show()


In [12]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from matplotlib.colors import hsv_to_rgb
from matplotlib.patches import Circle

# ----------------------------
# Example Inputs (ensure masks/img are defined)
# ----------------------------
# masks = ROIs.copy().astype(bool) # (N, H, W)
# img_rgb = img                  # (H, W, 3)
N, H, W = masks.shape

def generate_colors(n):
    if n == 0: return np.array([])
    return hsv_to_rgb(np.column_stack([
        np.linspace(0, 1, n, endpoint=False),
        np.ones(n),
        np.ones(n)
    ]))

colors = generate_colors(N)
ALPHA = 0.35

# Store references to prevent garbage collection
persistent_widgets = []

# ----------------------------
# Main Viewer
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 8))
plt.subplots_adjust(bottom=0.15)
ax.set_title("Cell Mask Viewer")

extent = [0, W, H, 0]
ax.imshow(img_rgb, extent=extent, interpolation='none')
overlay = np.zeros((H, W, 4), dtype=float)
im_overlay = ax.imshow(overlay, extent=extent, interpolation='none', zorder=10)
ax.axis('off')

hovered = None
create_mode = {"active": False}

def redraw_overlay(highlight=None):
    global colors
    overlay[:] = 0
    if len(colors) != len(masks):
        colors = generate_colors(len(masks))
    for i in range(len(masks)):
        m = masks[i]
        overlay[m > 0, :3] = colors[i]
        overlay[m > 0, 3] = 0.8 if i == highlight else ALPHA
    im_overlay.set_data(overlay)
    fig.canvas.draw_idle()

# Interaction for main window
def on_move(event):
    global hovered
    if event.inaxes != ax or create_mode["active"]: return
    if event.xdata is None or event.ydata is None: return
    x, y = int(event.xdata), int(event.ydata)
    if not (0 <= x < W and 0 <= y < H): return
    hits = np.where([masks[i, y, x] for i in range(len(masks))])[0]
    new_hover = hits[-1] if len(hits) > 0 else None
    if new_hover != hovered:
        hovered = new_hover
        redraw_overlay(hovered)

def on_click(event):
    global masks
    if event.inaxes != ax: return
    x, y = int(event.xdata), int(event.ydata)
    if create_mode["active"]:
        new_mask = np.zeros((H, W), dtype=bool)
        masks = np.concatenate([masks, new_mask[None, :, :]], axis=0)
        create_mode["active"] = False
        ax.set_title("Cell Mask Viewer")
        open_editor(len(masks) - 1, center=(y, x))
        return
    if hovered is not None:
        open_editor(hovered)

fig.canvas.mpl_connect("motion_notify_event", on_move)
fig.canvas.mpl_connect("button_press_event", on_click)

new_ax = plt.axes([0.4, 0.02, 0.2, 0.05])
new_btn = Button(new_ax, "Add New Mask")
new_btn.on_clicked(lambda e: [create_mode.update({"active": True}), 
                             ax.set_title("CLICK TO PLACE NEW MASK"), 
                             fig.canvas.draw_idle()])
persistent_widgets.append(new_btn)

# ----------------------------
# Side-by-Side Editor
# ----------------------------
def open_editor(idx, center=None):
    global masks
    mask = masks[idx]
    
    # Calculate crop region (80x80)
    if center is None and mask.any():
        ys, xs = np.where(mask)
        cy, cx = int(np.mean(ys)), int(np.mean(xs))
    else:
        cy, cx = center if center else (H//2, W//2)
    r = 40
    y0, y1 = max(0, cy-r), min(H, cy+r)
    x0, x1 = max(0, cx-r), min(W, cx+r)
    
    sub_img = img_rgb[y0:y1, x0:x1]
    sub_mask = mask[y0:y1, x0:x1].copy()

    # Create UI: Red vs Blue Channels
    fig2, (ax_r, ax_b) = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
    plt.subplots_adjust(bottom=0.25)
    
    ax_r.imshow(sub_img[:,:,0], cmap='gray', interpolation='none')
    ax_r.set_title("Red Channel (High Contrast)")
    ax_b.imshow(sub_img[:,:,2], cmap='gray', interpolation='none')
    ax_b.set_title("Blue Channel (High Contrast)")

    # Cyan Mask Overlay (30% Opaque)
    def get_cyan_overlay(m):
        overlay = np.zeros((m.shape[0], m.shape[1], 4))
        overlay[m > 0] = [0, 1, 1, 0.3] # Cyan: R=0, G=1, B=1
        return overlay

    im_r = ax_r.imshow(get_cyan_overlay(sub_mask), interpolation='none', zorder=5)
    im_b = ax_b.imshow(get_cyan_overlay(sub_mask), interpolation='none', zorder=5)
    
    # --- BRUSH CURSOR (Ghost Patches) ---
    # Create very thin white circles that follow the mouse
    cursor_r = Circle((0,0), radius=2, color='white', fill=False, lw=0.5, zorder=10)
    cursor_b = Circle((0,0), radius=2, color='white', fill=False, lw=0.5, zorder=10)
    ax_r.add_patch(cursor_r)
    ax_b.add_patch(cursor_b)

    for a in [ax_r, ax_b]: a.axis('off')

    state = {"drawing": False, "erase": False}
    slider_ax = plt.axes([0.2, 0.1, 0.6, 0.03])
    brush_slider = Slider(slider_ax, 'Brush Size', 0.5, 15.0, valinit=3.0)

    def paint_and_move(event):
        if event.inaxes not in [ax_r, ax_b]:
            cursor_r.set_visible(False)
            cursor_b.set_visible(False)
            fig2.canvas.draw_idle()
            return
        
        # Update cursor position and visibility
        cursor_r.set_visible(True)
        cursor_b.set_visible(True)
        cursor_r.center = (event.xdata, event.ydata)
        cursor_b.center = (event.xdata, event.ydata)
        
        # Update cursor radius from slider
        br = brush_slider.val
        cursor_r.set_radius(br)
        cursor_b.set_radius(br)

        if state["drawing"]:
            ix, iy = int(event.xdata), int(event.ydata)
            yy, xx = np.ogrid[-br:br+1, -br:br+1]
            circle_mask = xx**2 + yy**2 <= br**2
            
            sy0, sy1 = max(0, iy-int(br)), min(sub_mask.shape[0], iy+int(br)+1)
            sx0, sx1 = max(0, ix-int(br)), min(sub_mask.shape[1], ix+int(br)+1)
            
            # Slice circle to fit boundaries
            mask_chunk = circle_mask[:(sy1-sy0), :(sx1-sx0)]
            sub_mask[sy0:sy1, sx0:sx1][mask_chunk] = not state["erase"]
            
            new_overlay = get_cyan_overlay(sub_mask)
            im_r.set_data(new_overlay)
            im_b.set_data(new_overlay)
        
        fig2.canvas.draw_idle()

    # Event Connections
    fig2.canvas.mpl_connect("motion_notify_event", paint_and_move)
    fig2.canvas.mpl_connect("button_press_event", lambda e: [state.update({"drawing": True}), paint_and_move(e)])
    fig2.canvas.mpl_connect("button_release_event", lambda e: state.update({"drawing": False}))
    fig2.canvas.mpl_connect("key_press_event", lambda e: [state.update({"erase": not state["erase"]}) if e.key=='e' else None])


    # Save/Delete Buttons
    btn_save_ax = plt.axes([0.3, 0.03, 0.15, 0.06])
    btn_save = Button(btn_save_ax, "Save")
    def save_action(e):
        masks[idx][y0:y1, x0:x1] = sub_mask
        redraw_overlay()
        plt.close(fig2)
    btn_save.on_clicked(save_action)

    btn_del_ax = plt.axes([0.55, 0.03, 0.15, 0.06])
    btn_del = Button(btn_del_ax, "Delete", color='salmon')
    def del_action(e):
        global masks
        masks = np.delete(masks, idx, axis=0)
        redraw_overlay()
        plt.close(fig2)
    btn_del.on_clicked(del_action)

    # Keep widgets alive
    persistent_widgets.extend([brush_slider, btn_save, btn_del])
    plt.show()

redraw_overlay()
plt.show()


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].


In [14]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from matplotlib.colors import hsv_to_rgb
from matplotlib.patches import Circle

# ----------------------------
# Example Inputs (ensure masks/img are defined)
# ----------------------------
# masks = ROIs.copy().astype(bool) # (N, H, W)
# img_rgb = img                  # (H, W, 3)
N, H, W = masks.shape

def generate_colors(n):
    if n == 0: return np.array([])
    return hsv_to_rgb(np.column_stack([
        np.linspace(0, 1, n, endpoint=False),
        np.ones(n),
        np.ones(n)
    ]))

colors = generate_colors(N)
ALPHA = 0.35

# Store references to prevent garbage collection
persistent_widgets = []

# ----------------------------
# Integrated Main Viewer (Fixed Scaling)
# ----------------------------
fig, ax = plt.subplots(figsize=(10, 8))
plt.subplots_adjust(bottom=0.25) 
ax.set_title("Cell Mask Viewer")

extent = [0, W, H, 0]
# Ensure image is float for math; if it's 0-255, we normalize to 0-1
if img_rgb.max() > 1:
    img_orig = img_rgb.astype(float) / 255.0
else:
    img_orig = img_rgb.copy().astype(float)

im_base = ax.imshow(img_orig, extent=extent, interpolation='none')

overlay = np.zeros((H, W, 4), dtype=float)
im_overlay = ax.imshow(overlay, extent=extent, interpolation='none', zorder=10)
ax.axis('off')

# Global States
hovered = None
create_mode = {"active": False}
# Initialize at 1.0 (original appearance)
ui_state = {"masks_visible": True, "rg": 1.0, "b": 1.0}

def update_image_display():
    new_img = img_orig.copy()
    # Apply combined Red/Green slider and separate Blue slider
    new_img[..., 0] *= ui_state["rg"] # Red
    new_img[..., 1] *= ui_state["rg"] # Green
    new_img[..., 2] *= ui_state["b"]  # Blue
    im_base.set_data(np.clip(new_img, 0, 1))
    fig.canvas.draw_idle()

def redraw_overlay(highlight=None):
    global colors
    overlay[:] = 0
    if len(colors) != len(masks):
        colors = generate_colors(len(masks))
    
    if ui_state["masks_visible"]:
        for i in range(len(masks)):
            m = masks[i]
            overlay[m > 0, :3] = colors[i]
            # Highlight hovered mask or use standard alpha
            overlay[m > 0, 3] = 0.8 if i == highlight else ALPHA
            
    im_overlay.set_data(overlay)
    fig.canvas.draw_idle()

# --- Interaction Logic (Clicking/Hovering) ---
def on_move(event):
    global hovered
    if event.inaxes != ax or create_mode["active"]: return
    if event.xdata is None or event.ydata is None: return
    x, y = int(event.xdata), int(event.ydata)
    if not (0 <= x < W and 0 <= y < H): return
    
    # Check masks for hits (using list comprehension for safety)
    hits = [i for i in range(len(masks)) if masks[i, y, x]]
    new_hover = hits[-1] if hits else None
    
    if new_hover != hovered:
        hovered = new_hover
        redraw_overlay(hovered)

def on_click(event):
    global masks
    if event.inaxes != ax: return
    x, y = int(event.xdata), int(event.ydata)
    
    if create_mode["active"]:
        new_mask = np.zeros((H, W), dtype=bool)
        masks = np.concatenate([masks, new_mask[None, :, :]], axis=0)
        create_mode["active"] = False
        ax.set_title("Cell Mask Viewer")
        open_editor(len(masks) - 1, center=(y, x))
        return
    
    if hovered is not None:
        open_editor(hovered)

fig.canvas.mpl_connect("motion_notify_event", on_move)
fig.canvas.mpl_connect("button_press_event", on_click)

# --- UI Controls (Sliders & Buttons) ---
ax_rg = plt.axes([0.1, 0.1, 0.25, 0.03])
ax_b = plt.axes([0.1, 0.05, 0.25, 0.03])
ax_new = plt.axes([0.45, 0.07, 0.15, 0.06])
ax_toggle = plt.axes([0.65, 0.07, 0.15, 0.06])

# Range 0 to 1, starting at 1
sld_rg = Slider(ax_rg, 'R+G', 0.0, 1.0, valinit=1.0)
sld_b = Slider(ax_b, 'Blue', 0.0, 1.0, valinit=1.0)
btn_new = Button(ax_new, "Add New Mask")
btn_toggle = Button(ax_toggle, "Show/Hide Masks")

# Bindings
sld_rg.on_changed(lambda v: [ui_state.update({"rg": v}), update_image_display()])
sld_b.on_changed(lambda v: [ui_state.update({"b": v}), update_image_display()])
btn_new.on_clicked(lambda e: [create_mode.update({"active": True}), 
                             ax.set_title("CLICK IMAGE TO PLACE MASK"), 
                             fig.canvas.draw_idle()])
btn_toggle.on_clicked(lambda e: [ui_state.update({"masks_visible": not ui_state["masks_visible"]}), 
                                redraw_overlay()])

# Reference to keep widgets alive
main_widgets = [sld_rg, sld_b, btn_new, btn_toggle]


# ----------------------------
# Side-by-Side Editor
# ----------------------------
def open_editor(idx, center=None):
    global masks
    mask = masks[idx]
    
    # Calculate crop region (80x80)
    if center is None and mask.any():
        ys, xs = np.where(mask)
        cy, cx = int(np.mean(ys)), int(np.mean(xs))
    else:
        cy, cx = center if center else (H//2, W//2)
    r = 40
    y0, y1 = max(0, cy-r), min(H, cy+r)
    x0, x1 = max(0, cx-r), min(W, cx+r)
    
    sub_img = img_rgb[y0:y1, x0:x1]
    sub_mask = mask[y0:y1, x0:x1].copy()

    # Create UI: Red vs Blue Channels
    fig2, (ax_r, ax_b) = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
    plt.subplots_adjust(bottom=0.25)
    
    ax_r.imshow(sub_img[:,:,0], cmap='gray', interpolation='none')
    ax_r.set_title("Red Channel (High Contrast)")
    ax_b.imshow(sub_img[:,:,2], cmap='gray', interpolation='none')
    ax_b.set_title("Blue Channel (High Contrast)")

    # Cyan Mask Overlay (30% Opaque)
    def get_cyan_overlay(m):
        overlay = np.zeros((m.shape[0], m.shape[1], 4))
        overlay[m > 0] = [0, 1, 1, 0.3] # Cyan: R=0, G=1, B=1
        return overlay

    im_r = ax_r.imshow(get_cyan_overlay(sub_mask), interpolation='none', zorder=5)
    im_b = ax_b.imshow(get_cyan_overlay(sub_mask), interpolation='none', zorder=5)
    
    # --- BRUSH CURSOR (Ghost Patches) ---
    # Create very thin white circles that follow the mouse
    cursor_r = Circle((0,0), radius=2, color='white', fill=False, lw=0.5, zorder=10)
    cursor_b = Circle((0,0), radius=2, color='white', fill=False, lw=0.5, zorder=10)
    ax_r.add_patch(cursor_r)
    ax_b.add_patch(cursor_b)

    for a in [ax_r, ax_b]: a.axis('off')

    state = {"drawing": False, "erase": False}
    slider_ax = plt.axes([0.2, 0.1, 0.6, 0.03])
    brush_slider = Slider(slider_ax, 'Brush Size', 0.5, 15.0, valinit=3.0)

    def paint_and_move(event):
        if event.inaxes not in [ax_r, ax_b]:
            cursor_r.set_visible(False)
            cursor_b.set_visible(False)
            fig2.canvas.draw_idle()
            return
        
        # Update cursor position and visibility
        cursor_r.set_visible(True)
        cursor_b.set_visible(True)
        cursor_r.center = (event.xdata, event.ydata)
        cursor_b.center = (event.xdata, event.ydata)
        
        # Update cursor radius from slider
        br = brush_slider.val
        cursor_r.set_radius(br)
        cursor_b.set_radius(br)

        if state["drawing"]:
            ix, iy = int(event.xdata), int(event.ydata)
            yy, xx = np.ogrid[-br:br+1, -br:br+1]
            circle_mask = xx**2 + yy**2 <= br**2
            
            sy0, sy1 = max(0, iy-int(br)), min(sub_mask.shape[0], iy+int(br)+1)
            sx0, sx1 = max(0, ix-int(br)), min(sub_mask.shape[1], ix+int(br)+1)
            
            # Slice circle to fit boundaries
            mask_chunk = circle_mask[:(sy1-sy0), :(sx1-sx0)]
            sub_mask[sy0:sy1, sx0:sx1][mask_chunk] = not state["erase"]
            
            new_overlay = get_cyan_overlay(sub_mask)
            im_r.set_data(new_overlay)
            im_b.set_data(new_overlay)
        
        fig2.canvas.draw_idle()

    # Event Connections
    fig2.canvas.mpl_connect("motion_notify_event", paint_and_move)
    fig2.canvas.mpl_connect("button_press_event", lambda e: [state.update({"drawing": True}), paint_and_move(e)])
    fig2.canvas.mpl_connect("button_release_event", lambda e: state.update({"drawing": False}))
    fig2.canvas.mpl_connect("key_press_event", lambda e: [state.update({"erase": not state["erase"]}) if e.key=='e' else None])


    # Save/Delete Buttons
    btn_save_ax = plt.axes([0.3, 0.03, 0.15, 0.06])
    btn_save = Button(btn_save_ax, "Save")
    def save_action(e):
        masks[idx][y0:y1, x0:x1] = sub_mask
        redraw_overlay()
        plt.close(fig2)
    btn_save.on_clicked(save_action)

    btn_del_ax = plt.axes([0.55, 0.03, 0.15, 0.06])
    btn_del = Button(btn_del_ax, "Delete", color='salmon')
    def del_action(e):
        global masks
        masks = np.delete(masks, idx, axis=0)
        redraw_overlay()
        plt.close(fig2)
    btn_del.on_clicked(del_action)

    # Keep widgets alive
    persistent_widgets.extend([brush_slider, btn_save, btn_del])
    plt.show()

redraw_overlay()
plt.show()
